# Donut OCR Evaluation (Dev Split)

This notebook evaluates Donut OCR on the Maltese dev crop dataset and reports CER metrics.

Run the cells in order from top to bottom.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from accelerate import Accelerator
from transformers import DonutProcessor, VisionEncoderDecoderModel

/home/user/anaconda3/envs/nomocrat-donut/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Donut Model

Initialize the processor and model, then move the model to the active device.

In [2]:
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
try:
    model = VisionEncoderDecoderModel.from_pretrained(
        "naver-clova-ix/donut-base-finetuned-cord-v2",
        use_safetensors=True,
    )
except OSError as e:
    raise RuntimeError(
        "Could not load safetensors weights for this model. "
        "Upgrade torch to >=2.6 or install a transformers version compatible with your torch build."
    ) from e

device = Accelerator().device
model.to(device)  # doctest: +IGNORE_RESULT

Loading weights: 100%|██████████| 483/483 [00:00<00:00, 4273.06it/s]


VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0): DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )

## 3. Build Test Crop Index

Load test JSON annotations, resolve crop image paths, and create a per-box dataframe.

In [3]:
# build crop index from the DEV split json
WORKSPACE = Path.cwd()
JSON_PATH = WORKSPACE / "mlt_data" / "ocr_data_v11_dev.json"
CROPS_DIR = WORKSPACE / "mlt_data" / "crops" / "dev"

print(f"JSON exists: {JSON_PATH.exists()} -> {JSON_PATH}")
print(f"Crops dir exists: {CROPS_DIR.exists()} -> {CROPS_DIR}")

with JSON_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, dict):
    records = raw.get("data", [])
elif isinstance(raw, list):
    records = raw
else:
    raise TypeError("Unsupported json structure for OCR dev data")

rows = []
for rec in records:
    page_obj = rec.get("page", {}) if isinstance(rec, dict) else {}
    page_fname = str(
        rec.get("page_fname", page_obj.get("page_fname", rec.get("image", rec.get("filename", ""))))
    )
    if not page_fname:
        continue

    page_stem = Path(page_fname).stem
    if "-" in page_stem:
        default_doc, default_page = page_stem.split("-", 1)
    else:
        default_doc, default_page = page_stem, ""

    doc_id = str(rec.get("document_id", page_obj.get("document_id", default_doc)))
    page_id = str(rec.get("page_id", page_obj.get("page_id", default_page)))

    boxes = rec.get("boxes", rec.get("bb", rec.get("bboxes", []))) if isinstance(rec, dict) else []
    for box_idx, box in enumerate(boxes):
        if isinstance(box, dict):
            gt_text = str(
                box.get("text", box.get("gt_text", box.get("transcription", box.get("value", ""))))
            )
        else:
            gt_text = str(box)

        crop_path = CROPS_DIR / doc_id / page_id / f"{page_stem}_box{box_idx:04d}.jpg"
        rows.append(
            {
                "page_fname": page_fname,
                "doc_id": doc_id,
                "page_id": page_id,
                "box_idx": box_idx,
                "gt_text": gt_text,
                "crop_path": str(crop_path),
                "exists": crop_path.exists(),
            }
        )

expected_cols = ["page_fname", "doc_id", "page_id", "box_idx", "gt_text", "crop_path", "exists"]
df_crops = pd.DataFrame(rows)
if df_crops.empty:
    df_crops = pd.DataFrame(columns=expected_cols)
else:
    df_crops = df_crops.sort_values(["page_fname", "box_idx"]).reset_index(drop=True)

print(f"Crop records parsed: {len(df_crops)}")
print(f"Missing crop images: {(~df_crops['exists']).sum() if 'exists' in df_crops.columns else 0}")
df_crops.head(12)

JSON exists: True -> /home/user/Desktop/Nomocrat/Evaluation/Donut/mlt_data/ocr_data_v11_dev.json
Crops dir exists: True -> /home/user/Desktop/Nomocrat/Evaluation/Donut/mlt_data/crops/dev
Crop records parsed: 587
Missing crop images: 0


,page_fname,doc_id,page_id,box_idx,gt_text,crop_path,exists
0,0003-001.jpg,0003,001,0,IT-TISĦIĦ TA’ REŻILJENZA PERMEZZ TA’ SISTEMI T...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
1,0003-001.jpg,0003,001,1,Gwida għall-Istabbiliment ta’ Kultura ta’ Komu...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
2,0003-001.jpg,0003,001,2,L-Aġenzija Ewropea għall-Ħtiġijiet Speċjali u ...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
3,0005-003.jpg,0005,003,0,2,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
4,0005-003.jpg,0005,003,1,Tradott minn: Tarcisio Zarb,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
5,0005-003.jpg,0005,003,2,L-istampa tal-faċċata: collage mix-xoghol ta’ ...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
6,0005-003.jpg,0005,003,3,L-Aġenzija Ewropea għall-Iżvilupp ta’ Edukazzj...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
7,0005-003.jpg,0005,003,4,ll-fehmiet espressi minn individwi f’dan id-do...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
8,0005-003.jpg,0005,003,5,Verżjonijiet elettroniċi ta’ dan ir-rapport b’...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True
9,0005-003.jpg,0005,003,6,ISBN: 978-87-92387-68-4 (Elettronika),/home/user/Desktop/Nomocrat/Evaluation/Donut/m...,True


## 4. Run Donut OCR

Run inference on the available test crops and collect predictions.

In [4]:
# run Donut OCR on test crops
task_prompt = "<s_cord-v2>"
decoder_input_ids = processor.tokenizer(
    task_prompt, add_special_tokens=False, return_tensors="pt"
).input_ids

MAX_CROPS = None  # set to an int for quick debugging
to_run = df_crops[df_crops["exists"]].copy() if "exists" in df_crops.columns else df_crops.copy()
if MAX_CROPS is not None:
    to_run = to_run.head(MAX_CROPS)

pred_rows = []
model.eval()
for row in to_run.itertuples(index=False):
    image = Image.open(row.crop_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values

    with torch.no_grad():
        outputs = model.generate(
            pixel_values.to(device),
            decoder_input_ids=decoder_input_ids.to(device),
            max_length=model.decoder.config.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(
        processor.tokenizer.pad_token, ""
    )
    sequence = re.sub(r"<.*?>", " ", sequence).strip()
    sequence = re.sub(r"\s+", " ", sequence)

    pred_rows.append(
        {
            "page_fname": row.page_fname,
            "box_idx": int(row.box_idx),
            "gt_text": row.gt_text,
            "pred_text": sequence,
            "crop_path": row.crop_path,
        }
    )

expected_cols = ["page_fname", "box_idx", "gt_text", "pred_text", "crop_path"]
df_ocr = pd.DataFrame(pred_rows)
if df_ocr.empty:
    df_ocr = pd.DataFrame(columns=expected_cols)

print(f"Evaluated crops: {len(df_ocr)}")
df_ocr.head(12)

Evaluated crops: 587


,page_fname,box_idx,gt_text,pred_text,crop_path
0,0003-001.jpg,0,IT-TISĦIĦ TA’ REŻILJENZA PERMEZZ TA’ SISTEMI T...,IT-TISHIH TA' REZILJENZA SISTEMI TA' EDUKAZZON...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
1,0003-001.jpg,1,Gwida għall-Istabbiliment ta’ Kultura ta’ Komu...,Gwida ghall-Istabbiliment ta' Kultura ta' fi-E...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
2,0003-001.jpg,2,L-Aġenzija Ewropea għall-Ħtiġijiet Speċjali u ...,L-Agenzija Ewropea ghall-Htigijiet Speciali u ...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
3,0005-003.jpg,0,2,,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
4,0005-003.jpg,1,Tradott minn: Tarcisio Zarb,Tradott minn: Tarcisio Zarb,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
5,0005-003.jpg,2,L-istampa tal-faċċata: collage mix-xoghol ta’ ...,L-istampa tal-faccata: collage mix-xoghol ta' ...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
6,0005-003.jpg,3,L-Aġenzija Ewropea għall-Iżvilupp ta’ Edukazzj...,L-Agenzia Ewropea ghall-Izvilupp ta Edukaizzon...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
7,0005-003.jpg,4,ll-fehmiet espressi minn individwi f’dan id-do...,Il-fehmiet espressi munn individwi fdan id-dok...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
8,0005-003.jpg,5,Verżjonijiet elettroniċi ta’ dan ir-rapport b’...,Verzionijiet elettronici ta' dan ir-apport b'2...,/home/user/Desktop/Nomocrat/Evaluation/Donut/m...
9,0005-003.jpg,6,ISBN: 978-87-92387-68-4 (Elettronika),ISBN: 978-87-92387-68-4 (Elettronika),/home/user/Desktop/Nomocrat/Evaluation/Donut/m...


## 5. Compute CER Metrics

Normalize text, compute edit distances, and summarize weighted and per-crop CER.

In [9]:
# CER evaluation and preview
def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]

if df_ocr.empty:
    print("No OCR predictions available. Run Cell 4 after confirming crop paths in Cell 3.")
    df_eval = pd.DataFrame(columns=[
        "page_fname", "box_idx", "gt_len", "edit_distance", "cer", "gt_text", "pred_text"
    ])
    df_eval.head(0)
else:
    df_eval = df_ocr.copy()
    df_eval["pred_text"] = df_eval["pred_text"].fillna("")
    df_eval["gt_text"] = df_eval["gt_text"].fillna("")
    df_eval["pred_norm"] = df_eval["pred_text"].map(normalize_text)
    df_eval["gt_norm"] = df_eval["gt_text"].map(normalize_text)
    df_eval["gt_len"] = df_eval["gt_norm"].str.len()

    df_eval["edit_distance"] = [
        levenshtein_distance(pred, gt)
        for pred, gt in zip(df_eval["pred_norm"], df_eval["gt_norm"])
]
    df_eval["cer"] = df_eval["edit_distance"] / df_eval["gt_len"].replace(0, pd.NA)

    weighted_cer = df_eval["edit_distance"].sum() / max(df_eval["gt_len"].sum(), 1)
    mean_cer = df_eval["cer"].dropna().mean()

    print(f"Evaluated crops: {len(df_eval)}")
    print(f"Weighted CER: {weighted_cer:.4f}")
    print(f"Mean CER (per-crop): {mean_cer:.4f}")

    df_eval[["page_fname", "box_idx", "gt_len", "edit_distance", "cer", "gt_text", "pred_text"]].sort_values(
        "cer", ascending=False
).reset_index(drop=True).head(20)

Evaluated crops: 587
Weighted CER: 0.2394
Mean CER (per-crop): 0.3172


## 6. Page Analysis

In [19]:
# Visualize crops with GT and prediction for a specific page, plus page-level weighted CER
from IPython.display import display, Markdown

PAGE_ID = "0050-015"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)

if MAX_SHOW is not None:
    page_rows = page_rows.head(MAX_SHOW)

print(f"Page: {target_page}")
print(f"Crops shown: {len(page_rows)}")

if page_rows.empty:
    print("No rows found. Check PAGE_ID and ensure OCR/CER cells were run first.")
else:
    total_edits = page_rows["edit_distance"].sum()
    total_gt_len = int(page_rows["gt_len"].sum())
    page_weighted_cer = total_edits / max(total_gt_len, 1)
    page_mean_cer = page_rows["cer"].dropna().mean()

    print(f"Weighted CER for page: {page_weighted_cer:.4f}")
    print(f"Mean CER for page (per-crop): {page_mean_cer:.4f}")
    print("=" * 80)

    # for _, row in page_rows.iterrows():
    #     crop_path = Path(row["crop_path"])

    #     display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    #     display(Image.open(crop_path))
    #     display(Markdown("**GT**"))
    #     print(str(row["gt_text"]))
    #     display(Markdown("**Prediction**"))
    #     print(str(row["pred_text"]))
    #     print("-" * 80)

Page: 0050-015.jpg
Crops shown: 1
Weighted CER for page: 1.0000
Mean CER for page (per-crop): 1.0000


In [20]:
# Excel-friendly TSV output: one row per BB with GT, Prediction, and CER columns

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)


def _clean_text(v):
    if v is None:
        return ""
    return join_lines(str(v).splitlines())

# Prefer the currently selected page; fall back to all evaluated rows.
if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text", "cer"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))
if "cer" not in out.columns:
    out["cer"] = pd.NA

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
    CER=out["cer"].map(lambda x: "" if pd.isna(x) else f"{float(x):.4f}"),
)[["value", "GT", "Prediction", "CER"]]

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))

value	GT	Prediction	CER
page:0050-015.jpg bb:0000	14		1.0000



## 7. Worst-Page CER Analysis
Automatically find the page with the highest weighted CER and analyze it using the same page-level view as Section 7.

In [18]:
from IPython.display import display, Markdown

page_summary = (
    df_eval.groupby("page_fname", dropna=False)
    .agg(
        total_edits=("edit_distance", "sum"),
        total_gt_len=("gt_len", "sum"),
        mean_cer=("cer", "mean"),
        num_boxes=("box_idx", "count"),
    )
    .reset_index()
)
page_summary["weighted_cer"] = page_summary["total_edits"] / page_summary["total_gt_len"].clip(lower=1)
page_summary = page_summary.sort_values(["weighted_cer", "mean_cer", "page_fname"], ascending=[False, False, True]).reset_index(drop=True)

worst_page = str(page_summary.loc[0, "page_fname"])
worst_page_rows = df_eval[df_eval["page_fname"] == worst_page].sort_values("box_idx").reset_index(drop=True)
worst_page_weighted_cer = float(page_summary.loc[0, "weighted_cer"])
worst_page_mean_cer = float(page_summary.loc[0, "mean_cer"])
worst_page_boxes = int(page_summary.loc[0, "num_boxes"])

print(f"Worst page by weighted CER: {worst_page}")
print(f"Crops shown: {worst_page_boxes}")
print(f"Weighted CER for page: {worst_page_weighted_cer:.4f}")
print(f"Mean CER for page (per-crop): {worst_page_mean_cer:.4f}")
print("=" * 80)

for _, row in worst_page_rows.iterrows():
    crop_path = Path(row["crop_path"])

    # display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    # display(Image.open(crop_path))
    # display(Markdown("**GT**"))
    # print(str(row["gt_text"]))
    # display(Markdown("**Prediction**"))
    # print(str(row["pred_text"]))
    # print("-" * 80)

Worst page by weighted CER: 0050-015.jpg
Crops shown: 1
Weighted CER for page: 1.0000
Mean CER for page (per-crop): 1.0000
